In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PATH_DRIVE = '/content/drive/MyDrive/Eafit/TDG/Codigo/Final/'


OUTPUT_CSV = PATH_DRIVE + "silver_layer_13.csv"
champs = PATH_DRIVE + "all_champions.json"

print(f"Ruta de dataset: {OUTPUT_CSV}")
print(f"Ruta de campeones: {champs}")

Ruta de dataset: /content/drive/MyDrive/Eafit/TDG/Codigo/Final/silver_layer_13.csv
Ruta de campeones: /content/drive/MyDrive/Eafit/TDG/Codigo/Final/all_champions.json


Ajuste de silver

In [ ]:
import pandas as pd
import json
import re

# 1. Cargar Bronze
with open(champs, 'r') as f: # Asegúrate de que la ruta sea correcta
    bronze_data = json.load(f)

def get_riot_stat(base, growth, lvl):
    if lvl == 1: return base
    # Fórmula de crecimiento de Riot
    f = (lvl - 1) * (0.7025 + 0.0175 * (lvl - 1))
    return base + (growth * f)

def extract_detailed_spell(ability, skill_lvl):
    res = {'base': 0, 'r_ad': 0, 'r_ap': 0, 'r_hp': 0, 'type': 'OTHER', 'shield_base': 0, 'as_buff': 0}
    if not ability: return res
    res['type'] = str(ability.get('damageType', 'OTHER')).upper()

    for effect in ability.get('effects', []):
        for lev in effect.get('leveling', []):
            attr = lev.get('attribute', '').lower()
            mods = lev.get('modifiers', [])
            if not mods: continue
            for mod in mods:
                vals = mod.get('values', [0])
                # Evita error de índice si la habilidad tiene menos niveles (ej. la R)
                val = vals[min(skill_lvl - 1, len(vals) - 1)]
                unit = str(mod.get('units', [''])[0] if mod.get('units') else '').lower()

                if 'damage' in attr:
                    if unit in ['', 'none']: res['base'] = val
                    elif '% ad' in unit: res['r_ad'] = val / 100
                    elif '% ap' in unit: res['r_ap'] = val / 100
                    elif 'health' in unit: res['r_hp'] = val / 100
                if 'shield' in attr or 'heal' in attr:
                    if unit in ['', 'none']: res['shield_base'] = max(res['shield_base'], val)
                if 'attack speed' in attr and '%' in unit:
                    res['as_buff'] = max(res['as_buff'], val / 100)
    return res

# Diccionarios de CC para detección de texto
keywords_hard = ["stun", "root", "immob", "suppres", "sleep", "fear", "taunt", "airborne", "knock", "pulveriz", "charm", "flee"]
keywords_soft = ["slow", "blind", "silenc", "drowsy", "cripple"]

# 2. Procesamiento enfocado en NIVEL 13
# Reparto típico a nivel 13: 5 puntos a la principal, 3 a las secundarias, 2 a la ulti.
SKILL_DIST = {'Q': 5, 'W': 1, 'E': 5, 'R': 2}
LVL = 13

silver_rows = []

for name, champ in bronze_data.items():
    # Extraemos el ID numérico que Riot guarda en 'key'
    row = {'id': champ['id'],'key_name': champ['key'], 'name': name}
    s = champ['stats']

    # Stats Base al Nivel 13
    row[f'HP_{LVL}'] = round(get_riot_stat(s['health']['flat'], s['health']['perLevel'], LVL), 2)
    row[f'AD_{LVL}'] = round(get_riot_stat(s['attackDamage']['flat'], s['attackDamage']['perLevel'], LVL), 2)
    row[f'AR_{LVL}'] = round(get_riot_stat(s['armor']['flat'], s['armor']['perLevel'], LVL), 2)
    row[f'MR_{LVL}'] = round(get_riot_stat(s['magicResistance']['flat'], s['magicResistance']['perLevel'], LVL), 2)

    f_as = (LVL - 1) * (0.7025 + 0.0175 * (LVL - 1))
    row[f'AS_{LVL}'] = round(s['attackSpeed']['flat'] + (s['attackSpeedRatio']['flat'] * (s['attackSpeed']['perLevel']/100) * f_as), 3)

    max_as_buff = 0
    total_hard_cc = 0
    total_soft_cc = 0

    for slot in ['Q', 'W', 'E', 'R']:
        ability_list = champ['abilities'].get(slot, [])
        ability = ability_list[0] if ability_list else {}

        row[f'{slot}_range'] = ability.get('targetRange', 0)

        # Lógica de CC con búsqueda de palabras clave
        blurb = str(ability.get('blurb') or '').lower()
        notes = str(ability.get('notes') or '').lower()
        desc = blurb + " " + notes

        is_hard = any(kw in desc for kw in keywords_hard)
        is_soft = any(kw in desc for kw in keywords_soft)

        dur_match = re.search(r'(\d+\.?\d*)\s*(s|sec|second)', desc)
        if dur_match:
            duration = float(dur_match.group(1))
        else:
            if is_hard: duration = 1.25 # Buff de duración base para Apex Tier
            elif is_soft: duration = 1.5
            else: duration = 0.0

        row[f'{slot}_hard_cc_dur'] = duration if is_hard else 0.0
        row[f'{slot}_soft_cc_dur'] = duration if (not is_hard and is_soft) else 0.0

        total_hard_cc += row[f'{slot}_hard_cc_dur']
        total_soft_cc += row[f'{slot}_soft_cc_dur']

        # Daño y Ratios al Nivel 13
        pts = SKILL_DIST[slot]
        data = extract_detailed_spell(ability, pts)

        row[f'{slot}_base_dmg_{LVL}'] = data['base']
        row[f'{slot}_ratio_ad_{LVL}'] = data['r_ad']
        row[f'{slot}_ratio_ap_{LVL}'] = data['r_ap']
        row[f'{slot}_ratio_hp_{LVL}'] = data['r_hp']
        row[f'{slot}_damage_type'] = data['type']
        row[f'{slot}_shield_base_{LVL}'] = data['shield_base']

        max_as_buff = max(max_as_buff, data['as_buff'])

    # Metadatos finales
    row['total_hard_cc_dur'] = total_hard_cc
    row['total_soft_cc_dur'] = total_soft_cc
    row['as_buff_passive'] = max_as_buff
    row['range_auto'] = s['attackRange']['flat']

    silver_rows.append(row)

df_silver = pd.DataFrame(silver_rows)
df_silver.to_csv('capa_silver_lvl13.csv', index=False)

print(f"Capa Silver generada exitosamente enfocada en Nivel {LVL}.")
print(f"Total de campeones procesados: {len(df_silver)}")

Capa Silver generada exitosamente enfocada en Nivel 13.
Total de campeones procesados: 172


In [ ]:
df_silver.head(13)

,id,key_name,name,HP_13,AD_13,AR_13,MR_13,AS_13,Q_range,Q_hard_cc_dur,...,R_base_dmg_13,R_ratio_ad_13,R_ratio_ap_13,R_ratio_hp_13,R_damage_type,R_shield_base_13,total_hard_cc_dur,total_soft_cc_dur,as_buff_passive,range_auto
0,266,Aatrox,Aatrox,1898.30,114.75,90.56,54.45,0.829,None,1.25,...,0.0,0.3,0.0000,0.0,NONE,0,1.25,1.5,0.0,175
1,103,Ahri,Ahri,1728.80,85.85,66.99,44.23,0.819,None,0.00,...,125.0,0.0,0.3500,0.0,MAGIC_DAMAGE,0,1.25,0.0,0.0,550
2,84,Akali,Akali,1903.05,98.14,74.47,59.45,0.844,500 • 120,0.00,...,420.0,0.0,0.9000,0.0,MAGIC_DAMAGE,0,0.00,1.5,0.0,125
3,166,Akshan,Akshan,1781.65,84.85,77.47,44.23,0.813,None,0.00,...,630.0,2.7,0.0000,0.0,PHYSICAL_DAMAGE,0,4.00,0.0,0.0,500
4,12,Alistar,Alistar,1999.00,103.06,91.47,54.45,0.770,None,1.25,...,0.0,0.0,0.0000,0.0,NONE,0,1202.50,0.0,0.0,125
5,799,Ambessa,Ambessa,1834.50,95.85,88.66,54.45,0.796,None,0.00,...,250.0,0.0,0.0000,0.0,PHYSICAL_DAMAGE,0,1.25,1.5,0.0,125
6,32,Amumu,Amumu,1714.30,98.61,76.80,54.45,0.888,None,1.25,...,300.0,0.0,0.8000,0.0,MAGIC_DAMAGE,0,2.50,1.5,0.0,125
7,34,Anivia,Anivia,1557.40,86.04,70.28,44.23,0.773,None,0.00,...,67.5,0.0,0.1875,0.0,MAGIC_DAMAGE,0,1.25,3.0,0.0,600
8,1,Annie,Annie,1611.20,79.02,66.80,44.23,0.703,625,0.00,...,275.0,0.0,0.7500,0.0,MAGIC_DAMAGE,0,0.00,0.0,0.0,625
9,523,Aphelios,Aphelios,1716.90,80.19,71.99,44.23,0.816,None,1.25,...,0.0,0.0,0.0000,0.0,PHYSICAL_DAMAGE,0,1.25,1.0,0.0,550


In [ ]:
roles = "/content/drive/MyDrive/Eafit/TDG/Codigo/Final/roles_26_8.csv"

In [ ]:
roles_df = pd.read_csv(roles, sep=';')
roles_df.head()

,champion,class_1,Class_2,Role_1,Role_2
0,Aatrox,juggernaut,NaN,Top,Jungle
1,Ahri,Burst,NaN,Mid,NaN
2,Akali,Assassin,NaN,Mid,Top
3,Akshan,Marksman,Assassin,Mid,NaN
4,Alistar,Vanguard,NaN,Support,NaN


In [ ]:
# Unir tu tabla de clases con los datos de habilidades
df_silver_full = pd.merge(df_silver, roles_df, left_on='key_name', right_on='champion', how='left')

In [ ]:
df_silver_full.head()

,id,key_name,name,HP_13,AD_13,AR_13,MR_13,AS_13,Q_range,Q_hard_cc_dur,...,R_shield_base_13,total_hard_cc_dur,total_soft_cc_dur,as_buff_passive,range_auto,champion,class_1,Class_2,Role_1,Role_2
0,266,Aatrox,Aatrox,1898.30,114.75,90.56,54.45,0.829,None,1.25,...,0,1.25,1.5,0.0,175,Aatrox,juggernaut,NaN,Top,Jungle
1,103,Ahri,Ahri,1728.80,85.85,66.99,44.23,0.819,None,0.00,...,0,1.25,0.0,0.0,550,Ahri,Burst,NaN,Mid,NaN
2,84,Akali,Akali,1903.05,98.14,74.47,59.45,0.844,500 • 120,0.00,...,0,0.00,1.5,0.0,125,Akali,Assassin,NaN,Mid,Top
3,166,Akshan,Akshan,1781.65,84.85,77.47,44.23,0.813,None,0.00,...,0,4.00,0.0,0.0,500,Akshan,Marksman,Assassin,Mid,NaN
4,12,Alistar,Alistar,1999.00,103.06,91.47,54.45,0.770,None,1.25,...,0,1202.50,0.0,0.0,125,Alistar,Vanguard,NaN,Support,NaN


In [ ]:
df_silver_full.to_csv('capa_silver_final_full_13.csv', index=False)
print("Capa Silver generada exitosamente.")

Capa Silver generada exitosamente.
